In [3]:
import random
import numpy as np

In [4]:
def sigmoid(z):
    return 1.0/(1.0+np.exp(-z))

def sigmoid_prime(z):
    return sigmoid(z)*(1-sigmoid(z))

In [5]:


class Network:
    def __init__(self, sizes):
        self.n_layers = len(sizes)
        self.sizes = sizes
        self.biases = [np.random.randn(y, 1) for y in sizes[1:]]
        self.weights = [np.random.randn(y, x) for x,y in zip(sizes[:-1], sizes[1:])]


    def feedforward(self, a):
        for b, w in zip(self.biases, self.weights):
            a = sigmoid(np.dot(w, a)+b)

        return a

    def SGD(self, training_data, epochs, mini_batch_size, eta, test_data=None):
        if test_data: n_test = len(test_data)
        n = len(training_data)
        for j in range(epochs):
            random.shuffle(training_data)
            mini_batches = [training_data[k:k+mini_batch_size] for k in range(0, n, mini_batch_size)]
            for mini_batch in mini_batches:
                self.update_mini_batch(mini_batch, eta)
            if test_data:
                print("Epoch {0}: {1} / {2}".format(j, self.evaluate(test_data), n_test))
            else:
                print("Epoch {0} complete".format(j))

    def update_mini_batch(self, mini_batch, eta):
        nabla_b = [np.zeros(b.shape) for b in self.biases]
        nabla_w = [np.zeros(w.shape) for w in self.weights]

        for x,y in mini_batch:
            delta_nabla_b, delta_nabla_w = self.backprop(x, y)
            nabla_b = [nb + dnb for nb, dnb in zip(nabla_b, delta_nabla_b)]
            nabla_w = [nw + dnw for nw, dnw in zip(nabla_w, delta_nabla_w)]

        self.weights = [w - (eta/len(mini_batch))*nw for w, nw in zip(self.weights, nabla_w)]
        self.biases = [b - (eta/len(mini_batch))*nb for b, nb in zip(self.biases, nabla_b)]

    def backprop(self, x, y):
        nabla_b = [np.zeros(b.shape) for b in self.biases]
        nabla_w = [np.zeros(w.shape) for w in self.weights]

        activation = x
        activations = [x]
        zs = []
        for b, w in zip(self.biases, self.weights):
            z = np.dot(w, activation) + b
            zs.append(z)
            activation = sigmoid(z)
            activations.append(activation)

        delta = self.cost_derivative(activations[-1], y) * sigmoid_prime(zs[-1])

        
        nabla_b[-1] = delta
        nabla_w[-1] = np.dot(delta, activations[-2].transpose())

        for l in range(2, self.n_layers):
            z = zs[-l]
            sp = sigmoid_prime(z)
            delta = np.dot(self.weights[-l+1].transpose(), delta) * sp
            nabla_b[-l] = delta
            nabla_w[-l] = np.dot(delta, activations[-l-1].transpose())

        return (nabla_b, nabla_w)

    def evaluate(self, test_data):
        test_results = [(np.argmax(self.feedforward(x)), y) for (x, y) in test_data]
        return sum(int(x == y) for (x, y) in test_results)

    def cost_derivative(self, output_activation, y):
        return (output_activation-y)





In [6]:
import sys
sys.path.append('/Users/michael/ml-course-inno/src')
sys.path.append('/Users/michael/ml-course-inno/data')
import mnist_loader

training_data, validation_data, test_data = mnist_loader.load_data_wrapper()

# net = Network([784, 30, 10])
# net.SGD(training_data, 30, 10, 3.0, test_data=test_data)

In [22]:
class TensorNetwork:
    def __init__(self, sizes):
        self.n_layers = len(sizes)
        self.sizes = sizes
        self.biases = [np.random.randn(y, 1) for y in sizes[1:]]
        self.weights = [np.random.randn(y, x) for x, y in zip(sizes[:-1], sizes[1:])]


    def feedforward(self, a):
        for w, b in zip(self.weights, self.biases):
            a = sigmoid(np.dot(w, a) + b)
        return a

    def SGD(self, training_data, epochs, eta, mini_batch_size, test_data=None):
        training_data = list(training_data)
        n = len(training_data)
        for j in range(epochs):
            random.shuffle(training_data)
            mini_batches = [training_data[k:k+mini_batch_size] for k in range(0, n, mini_batch_size)]
            for mini_batch in mini_batches:
                self.update_mini_batch(mini_batch, eta)
            if test_data:
                test_data = list(test_data)
                print(f"Epoch {j}: {self.evaluate(test_data)} / {len(test_data)}")
            else:
                print(f"Epoch {j} complete")

    def update_mini_batch(self, mini_batch, eta):
        """Update weights and biases using matrix-based backprop over the entire mini-batch."""
        m = len(mini_batch)
        
        # Stack inputs
        x_list = [x_vec for x_vec, _ in mini_batch]
        y_list = [y_vec for _, y_vec in mini_batch]
        
        x = np.hstack(x_list)
        y = np.hstack(y_list)
        
        nabla_b, nabla_w = self.backprop_minibatch(x, y)
        
        self.weights = [w - (eta / len(mini_batch)) * nw 
                        for w, nw in zip(self.weights, nabla_w)]
        self.biases = [b - (eta / len(mini_batch)) * nb 
                       for b, nb in zip(self.biases, nabla_b)]

    def backprop_minibatch(self, x, y):
        """Return (nabla_b, nabla_w) — the summed gradients for the mini-batch."""
        nabla_b = [np.zeros(b.shape) for b in self.biases]
        nabla_w = [np.zeros(w.shape) for w in self.weights]
    
        
        # forward pass
        activations = [x]
        zs = []
        for i, (b, w) in enumerate(zip(self.biases, self.weights)):
            z = np.dot(w, activations[-1]) + b
            zs.append(z)
            activations.append(sigmoid(z))
        
        # backward pass
        delta = self.cost_derivative(activations[-1], y) * sigmoid_prime(zs[-1])
        
        nabla_b[-1] = delta.sum(axis=1, keepdims=True)
        nabla_w[-1] = np.dot(delta, activations[-2].T)
        
        for l in range(2, self.n_layers):
            z = zs[-l]
            sp = sigmoid_prime(z)
            delta = np.dot(self.weights[-l + 1].T, delta) * sp
            nabla_b[-l] = delta.sum(axis=1, keepdims=True)
            nabla_w[-l] = np.dot(delta, activations[-l - 1].T)
        
        return (nabla_b, nabla_w)

    def cost_derivative(self, output_activations, y):
        return (output_activations - y)

    def evaluate(self, test_data):
        test_results = [(np.argmax(self.feedforward(x)), y) for x, y in test_data]
        return sum(int(x == y) for (x,y) in test_results)

In [23]:
x_sample, y_sample = training_data[0]
print(f"x_sample type: {type(x_sample)}, shape: {np.array(x_sample).shape}")
print(f"y_sample type: {type(y_sample)}, value: {y_sample}")

x_sample type: <class 'numpy.ndarray'>, shape: (784, 1)
y_sample type: <class 'numpy.ndarray'>, value: [[0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [1.]
 [0.]
 [0.]
 [0.]
 [0.]]


In [25]:
tensor_net = TensorNetwork([784, 30, 10])
tensor_net.SGD(training_data=training_data, epochs=30, mini_batch_size=10, eta=3.0, test_data=test_data)

Epoch 0: 9070 / 10000
Epoch 1: 9212 / 10000
Epoch 2: 9315 / 10000
Epoch 3: 9373 / 10000
Epoch 4: 9366 / 10000
Epoch 5: 9391 / 10000
Epoch 6: 9420 / 10000
Epoch 7: 9411 / 10000
Epoch 8: 9434 / 10000
Epoch 9: 9434 / 10000
Epoch 10: 9466 / 10000
Epoch 11: 9462 / 10000
Epoch 12: 9454 / 10000
Epoch 13: 9446 / 10000
Epoch 14: 9424 / 10000
Epoch 15: 9489 / 10000
Epoch 16: 9469 / 10000
Epoch 17: 9472 / 10000
Epoch 18: 9449 / 10000
Epoch 19: 9494 / 10000
Epoch 20: 9481 / 10000
Epoch 21: 9496 / 10000
Epoch 22: 9499 / 10000
Epoch 23: 9489 / 10000
Epoch 24: 9491 / 10000
Epoch 25: 9497 / 10000
Epoch 26: 9515 / 10000
Epoch 27: 9528 / 10000
Epoch 28: 9506 / 10000
Epoch 29: 9503 / 10000
